# Đọc Dataset Oxford Flowers 102

Review cấu trúc dataset và ví dụ đọc bằng PyTorch ImageFolder.

## 1. Review cấu trúc dataset

### Chuẩn Oxford Flowers 102 (gốc từ VGG):
- `jpg/` – tất cả ảnh trong 1 thư mục (image_00001.jpg …)
- `imagelabels.mat` – nhãn lớp
- `setid.mat` – train/val/test split

### Chuẩn ImageFolder (PyTorch/Keras – phổ biến cho transfer learning):
```
data/
├── train/
│   ├── 1/    ← class_id = tên thư mục
│   │   ├── image_xxx.jpg
│   │   └── ...
│   ├── 2/
│   └── ... 102/
├── valid/
│   ├── 1/
│   └── ...
└── test/
    ├── 1/    ← phải có subfolder class
    └── ...
```

### Cấu trúc hiện tại của bạn:
| Split | Cấu trúc | Chuẩn? |
|-------|----------|--------|
| train | `train/1/`, `train/2/`, ... | ✅ Chuẩn ImageFolder |
| valid | `valid/1/`, `valid/2/`, ... | ✅ Chuẩn ImageFolder |
| test  | `test/image_xxx.jpg` (flat) | ⚠️ Chạy **Phần 5** để tổ chức lại |

**Sau khi chạy Phần 5:** test sẽ có cấu trúc `test/1/`, `test/2/`, ... `test/102/` giống train và valid.

## 2. Thống kê số lượng mẫu

In [1]:
from pathlib import Path

data_dir = Path("flower_data/flower_data")

for split in ["train", "valid", "test"]:
    folder = data_dir / split
    if not folder.exists():
        continue
    count = sum(1 for _ in folder.rglob("*.jpg"))
    print(f"{split}: {count} mẫu")

n_classes = len(list((data_dir / "train").iterdir())) if (data_dir / "train").exists() else 0
print(f"\nSố lớp: {n_classes}")

train: 6552 mẫu
valid: 818 mẫu
test: 819 mẫu

Số lớp: 102


## 3. Đọc dataset (ImageFolder + DataLoader)

In [2]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

data_dir = "flower_data/flower_data"
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
])

train_dataset = datasets.ImageFolder(f"{data_dir}/train", transform=transform)
valid_dataset = datasets.ImageFolder(f"{data_dir}/valid", transform=transform)
test_dataset = datasets.ImageFolder(f"{data_dir}/test", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f"Train: {len(train_dataset)} mẫu | Valid: {len(valid_dataset)} mẫu | Test: {len(test_dataset)} mẫu")
print(f"Số lớp: {len(train_dataset.classes)}")
images, labels = next(iter(train_loader))
print(f"Batch: images {images.shape}, labels {labels.shape}")

Train: 6552 mẫu | Valid: 818 mẫu
Số lớp: 102
Batch: images torch.Size([32, 3, 224, 224]), labels torch.Size([32])


## 4. Load nhãn tên (cat_to_name.json)

In [3]:
import json

with open("flower_data/flower_data/cat_to_name.json") as f:
    cat_to_name = json.load(f)

print(f"Ví dụ: class 1 → {cat_to_name['1']}, class 74 → {cat_to_name['74']}")

Ví dụ: class 1 → pink primrose, class 74 → rose


## 5. Tổ chức lại test thành subfolder class

Dùng `imagelabels.mat` từ Oxford VGG (nhãn cho tất cả 8189 ảnh) để map mỗi ảnh `image_XXXXX.jpg` → nhãn lớp, rồi di chuyển vào `test/1/`, `test/2/`, ... `test/102/`. Hoạt động với mọi split (kể cả split tùy chỉnh khác Oxford).

In [ ]:
import re
import shutil
from pathlib import Path
from urllib.request import urlretrieve

# Cần: pip install scipy
from scipy.io import loadmat

BASE = Path("flower_data/flower_data")
TEST_DIR = BASE / "test"
URL = "https://www.robots.ox.ac.uk/~vgg/data/flowers/102/"

# 1. Tải imagelabels.mat (chứa nhãn cho TẤT CẢ 8189 ảnh)
fname = "imagelabels.mat"
dest = BASE / fname
if not dest.exists():
    print(f"Đang tải {fname}...")
    urlretrieve(URL + fname, dest)
else:
    print(f"{fname} đã có sẵn")

# 2. Load: imagelabels có nhãn cho mỗi ảnh (index 0 = image_00001, index 1 = image_00002, ...)
labels_mat = loadmat(BASE / "imagelabels.mat", squeeze_me=True)
all_labels = labels_mat["labels"]  # 1-102, index i = ảnh image_{i+1:05d}.jpg

# 3. Di chuyển ảnh vào subfolder class (chỉ file trực tiếp trong test/)
moved = 0
for fpath in list(TEST_DIR.glob("*.jpg")):
    m = re.match(r"image_(\d+)\.jpg", fpath.name, re.I)
    if not m:
        continue
    img_id = int(m.group(1))
    if img_id < 1 or img_id > len(all_labels):
        print(f"Bỏ qua {fpath.name} (id ngoài phạm vi 1-{len(all_labels)})")
        continue
    label = int(all_labels[img_id - 1])  # 1-102
    dest_dir = TEST_DIR / str(label)
    dest_dir.mkdir(parents=True, exist_ok=True)
    shutil.move(str(fpath), str(dest_dir / fpath.name))
    moved += 1

print(f"\nĐã di chuyển {moved} ảnh vào test/1/, test/2/, ... test/102/")